# Session 5 - VLM main audit


**GPU T4 x2, 2-3 h per model, repeated until complete.** The main audit.

One process per card (`CUDA_VISIBLE_DEVICES`, shards `0/2` and `1/2`). Every model
call is cached by content, so a re-run continues rather than restarts; finished
samples are rebuilt from the cache without re-splicing.

**Run this notebook once per model.** Keep `TAG`, `SEED` and `VLM_SAMPLES` constant
across resumed runs of the same model, otherwise the cache does not line up.

In [ ]:
SESSION = "S5 VLM main"

# ============================== CONFIG ==============================
MODEL        = "qwen25vl:Qwen/Qwen2.5-VL-3B-Instruct"
QUANT        = "fp16"      # "4bit" for the 7B model
MAX_PIXELS   = 384 * 384
TAG          = "main"      # keep constant across resumed runs
VLM_SAMPLES  = 600         # keep constant across resumed runs
SEED         = 0           # keep constant across resumed runs
SPLIT        = "test"      # every reported number comes from TEST

PROMPT_VARIANTS = 5        # prompt-paraphrase stability; 0 = off
SPLICE_FLOOR    = True
USE_BOTH_GPUS   = True
TIME_BUDGET_MIN = 420      # 7 h; the runner stops cleanly and flushes its cache

In [ ]:
# ---------------------------------------------------------------- BOOT
# Locates the code dataset wherever it is mounted, puts it on sys.path,
# prints the attached inputs, and records provenance.  The search is by
# file name, so the dataset's mount name does not matter.
import os, sys, subprocess, json, time

def _find_code():
    for root in ("/kaggle/input", "."):
        if not os.path.isdir(root):
            continue
        for dirpath, dirnames, files in os.walk(root):
            dirnames[:] = [d for d in dirnames if not d.startswith(".")]
            if os.path.basename(dirpath) == "ccaudit" and "kaggle_utils.py" in files:
                return os.path.dirname(dirpath)
    raise FileNotFoundError(
        "Could not find the ccaudit package.\n"
        "Add Input -> your code dataset (<your-code-dataset>), and check that "
        "the preview shows ccaudit/kaggle_utils.py at the top level.")

CODE = _find_code()
if CODE not in sys.path:
    sys.path.insert(0, CODE)
# Child processes do not inherit sys.path.  Every `python -m ccaudit.<module>`
# below runs as a subprocess, so the code directory must be on PYTHONPATH.
os.environ["PYTHONPATH"] = CODE + os.pathsep + os.environ.get("PYTHONPATH", "")
from ccaudit import kaggle_utils as KU
from ccaudit import common as C

OUT = KU.work_dir("audit")
TMP = KU.temp_dir()
os.environ["HF_HOME"] = KU.temp_dir("hf")          # model weights stay out of /kaggle/working
os.environ["TOKENIZERS_PARALLELISM"] = "false"
KU.session_header(SESSION, OUT)
print("code:", CODE)

In [ ]:
# ------------------------------------------------------- SELF TEST (always)
# The self-test suite runs in under a minute and needs no dataset.  Each check
# corresponds to a failure mode that would produce plausible-looking but
# incorrect numbers, so a failure here invalidates everything that follows.
rc = KU.sh(f"{sys.executable} {CODE}/scripts/selftest.py", check=False)
if rc != 0:
    raise SystemExit("SELF TEST FAILED -- inspect the failures above before proceeding.")

In [ ]:
KU.pip_install("transformers accelerate qwen-vl-utils")
if QUANT == "4bit":
    KU.pip_install("bitsandbytes")
KU.gpu_report()
INDEX = KU.find_parsed_index()
if not INDEX:
    raise SystemExit("Add Input -> `cca-s1-parsed`.")
recs, meta = C.load_index(INDEX)
VOCAB = meta.get("vocab", "face8")

# Seed the cache from every attached previous run of this model.
prev = [d for d in KU.find_run_dirs() if "run_vlm" in d]
RESUME = ",".join(prev)
print("INDEX  =", INDEX)
print("resume =", RESUME or "(nothing attached; this is the first run)")

In [ ]:
# ==================== MAIN AUDIT (one process per GPU) ====================
n_gpu = KU.n_gpus() if USE_BOTH_GPUS else 1
n_gpu = max(1, n_gpu)
floor = " --splice-floor" if SPLICE_FLOOR else ""
res   = f' --resume-from "{RESUME}"' if RESUME else ""
cmds, envs, logs = [], [], []
for i in range(n_gpu):
    logs.append(f"{OUT}/logs/vlm/proc_{i}.log")
    envs.append({"CUDA_VISIBLE_DEVICES": str(i)})
    cmds.append(
        f'{sys.executable} -m ccaudit.m5_runner --index "{INDEX}" '
        f'--detector "{MODEL}" --out "{OUT}/run_vlm" --tag {TAG} '
        f'--split {SPLIT} --limit-samples {VLM_SAMPLES} --seed {SEED} '
        f'--shard {i}/{n_gpu} --device cuda --quant {QUANT} '
        f'--max-pixels {MAX_PIXELS} --prompt-variants {PROMPT_VARIANTS} '
        f'--vocab {VOCAB} --time-budget-min {TIME_BUDGET_MIN}{floor}{res}')
KU.run_parallel(cmds, envs=envs, logs=logs, check=False, poll_sec=60)

In [ ]:
# ==================== PROGRESS ====================
import glob
done = set()
for p in glob.glob(f"{OUT}/run_vlm/raw_*.json"):
    for r in C.load_json(p).get("rows", []):
        done.add(r["sample_id"])
print(f"VLM samples complete: {len(done)} / {VLM_SAMPLES}")
if len(done) < VLM_SAMPLES:
    print("\nNOT FINISHED. Make a dataset from this output (Output tab -> New "
          "Dataset, e.g. `cca-s5-vlm`), attach it to this notebook, and Save & "
          "Run All again. Finished samples are rebuilt from the cache in "
          "seconds; only the remainder costs GPU time.")
else:
    print("\nComplete. Make the dataset and continue.")
for l in logs:
    KU.tail(l, 6)

In [ ]:
# ==================== INTERIM METRICS ====================
KU.sh(f'{sys.executable} -m ccaudit.m6_metrics --raw "{OUT}/run_vlm" '
      f'--out "{OUT}/metrics_vlm" --split {SPLIT} --vocab {VOCAB}', check=False)
for r in C.load_json(f"{OUT}/metrics_vlm/metrics.json", {}).get("results", []):
    print(f"{r['detector']}: n={r.get('n_samples')} AUC={r.get('AUC'):.3f} "
          f"FS={r.get('FS'):.4f} [{r.get('FS_lo'):.4f},{r.get('FS_hi'):.4f}] "
          f"CR-prior={r.get('CR_minus_prior'):.3f} "
          f"faithful={r.get('faithful')}")

In [ ]:
NEXT_STEP = """1. Output tab -> New Dataset -> `cca-s5-vlm-<model>` (a few MB: cache + rows).
2. If `VLM samples complete` is below VLM_SAMPLES, attach that dataset back to
   this notebook and Save & Run All again. Repeat until complete.
3. Then change MODEL and repeat for the next model in the set.
4. Attach every `cca-s5-vlm-*` dataset to Session 11; overlapping runs are
   de-duplicated by content key, so attaching more than needed is safe."""

# ----------------------------------------------------------- WRAP UP
KU.disk_report()
print(C.banner("NEXT STEP"))
print(NEXT_STEP)